# Habitat 3.0 Benchmarking
Uses the **official `hab3_benchmark.py`** script from habitat-lab repo. Same configs as the Habitat 3.0 paper.

Measures Steps-Per-Second (SPS) for:
1. Single Spot robot with oracle nav
2. Single humanoid with oracle nav
3. Multi-agent (Spot + Spot) with oracle nav
4. Multi-agent (Spot + humanoid) with oracle nav

## 1. Setup

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv

if [ ! -d /content/habitat-lab-v033 ]; then
    cd /content
    git clone --depth 1 --branch v0.3.3 https://github.com/facebookresearch/habitat-lab.git habitat-lab-v033
    pip uninstall habitat-lab -y
    cd habitat-lab-v033
    pip install -e habitat-lab/
fi
python -c "import habitat; print('habitat:', habitat.__version__)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Symlinks from notebooks 08/09
mkdir -p /content/data/objects /content/data/robots

ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/hab3_bench_assets /content/data/hab3_bench_assets
ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/ycb /content/data/objects/ycb
ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/hab_spot_arm /content/data/robots/hab_spot_arm

# Also symlink data folder into habitat-lab-v033 (script expects relative paths)
ln -sfn /content/data /content/habitat-lab-v033/data 2>/dev/null || true

cat > /tmp/nvidia_egl.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF
echo "Ready"

## 2. Inspect the official benchmark configs

The repo has configs at `habitat-lab-v033/habitat-lab/habitat/config/benchmark/rearrange/hab3_bench/`

In [ ]:
%%bash
ls /content/habitat-lab-v033/habitat-lab/habitat/config/benchmark/rearrange/hab3_bench/

## 3. Benchmark 1: Single Spot robot with oracle nav
Same config as `hab3_bench/bench_runner.sh` uses: `spot_oracle.yaml` + `small_large.json.gz` dataset

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content/habitat-lab-v033
export MAGNUM_LOG=quiet
export HABITAT_SIM_LOG=quiet
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all
export OMP_NUM_THREADS=2

mkdir -p data/profile

python scripts/hab3_bench/hab3_benchmark.py \
    --cfg benchmark/rearrange/hab3_bench/spot_oracle.yaml \
    --n-steps 300 \
    --n-procs 1 \
    --out-name robot_oracle_trial1 \
    habitat.task.task_spec=tidy_house_2obj \
    habitat.task.pddl_domain_def=fp \
    habitat.dataset.data_path=data/hab3_bench_assets/episode_datasets/small_large.json.gz 2>&1 | tail -15

## 4. Benchmark 2: Single Humanoid with oracle nav
Uses `humanoid_oracle.yaml`

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content/habitat-lab-v033
export MAGNUM_LOG=quiet
export HABITAT_SIM_LOG=quiet
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all
export OMP_NUM_THREADS=2

python scripts/hab3_bench/hab3_benchmark.py \
    --cfg benchmark/rearrange/hab3_bench/humanoid_oracle.yaml \
    --n-steps 300 \
    --n-procs 1 \
    --out-name human_oracle_trial1 \
    habitat.task.task_spec=tidy_house_2obj \
    habitat.task.pddl_domain_def=fp \
    habitat.dataset.data_path=data/hab3_bench_assets/episode_datasets/small_large.json.gz 2>&1 | tail -15

## 5. Benchmark 3: Multi-agent Spot + Spot
Uses `spot_spot_oracle.yaml`

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content/habitat-lab-v033
export MAGNUM_LOG=quiet
export HABITAT_SIM_LOG=quiet
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all
export OMP_NUM_THREADS=2

python scripts/hab3_bench/hab3_benchmark.py \
    --cfg benchmark/rearrange/hab3_bench/spot_spot_oracle.yaml \
    --n-steps 300 \
    --n-procs 1 \
    --out-name robots_oracle_trial1 \
    habitat.task.pddl_domain_def=fp \
    habitat.dataset.data_path=data/hab3_bench_assets/episode_datasets/small_large.json.gz 2>&1 | tail -15

## 6. Benchmark 4: Multi-agent Spot + Humanoid
Uses `spot_humanoid_oracle.yaml` — Spot robot collaborating with humanoid

In [ ]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content/habitat-lab-v033
export MAGNUM_LOG=quiet
export HABITAT_SIM_LOG=quiet
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all
export OMP_NUM_THREADS=2

python scripts/hab3_bench/hab3_benchmark.py \
    --cfg benchmark/rearrange/hab3_bench/spot_humanoid_oracle.yaml \
    --n-steps 300 \
    --n-procs 1 \
    --out-name robot_human_oracle_trial1 \
    habitat.task.pddl_domain_def=fp \
    habitat.dataset.data_path=data/hab3_bench_assets/episode_datasets/small_large.json.gz 2>&1 | tail -15

## 7. Results

Each benchmark prints the FPS (frames/steps per second). Compare across configurations to see:
- **Single vs multi-agent overhead**
- **Robot (Spot) vs Humanoid cost**
- **Combined robot + human performance**

The official paper used dual Xeon Gold + 8×2080 Ti GPUs — Colab T4 will be significantly slower but relative differences should be consistent.